# Debate Analysis: Religion Does More Harm Than Good
This notebook analyzes the multi-agent debate transcript between PRO and CON. We explore token usage, argument word counts, persuasion score progressions, and cost metrics to understand the dynamics of agentic debate.

In [ ]:
import json
import os
import re
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Load data
transcript_path = "results/sample_debate.json"
if not os.path.exists(transcript_path):
    transcript_path = "../results/sample_debate.json" # Fallback if run inside notebooks/

with open(transcript_path, 'r') as f:
    data = json.load(f)

topic = data.get("topic")
rounds = data.get("rounds", [])
verdict = data.get("verdict", {})
total_cost = data.get("total_cost_usd", 0.0)
total_tokens = data.get("total_tokens", 0)

print(f"Loaded debate on topic: {topic}")
print(f"Total Rounds: {len(rounds)}")


In [ ]:
# Compute per-round metrics
round_numbers = []
pro_words = []
con_words = []
pro_tokens = []
con_tokens = []
pro_scores = []
con_scores = []
pro_evidence = []
con_evidence = []

def count_evidence(text):
    # Proxy for evidence: count numbers, percentages, and citations (e.g. "Smith (2007)")
    percs = len(re.findall(r'\d+%', text))
    dates = len(re.findall(r'\d{4}', text))
    dollars = len(re.findall(r'\$\d+', text))
    return percs + dates + dollars

for r in rounds:
    rn = r["round_number"]
    round_numbers.append(rn)
    
    p_msg = r["pro_message"]
    c_msg = r["con_message"]
    
    pw = len(p_msg.split())
    cw = len(c_msg.split())
    pro_words.append(pw)
    con_words.append(cw)
    
    pro_tokens.append(int(pw * 1.3))
    con_tokens.append(int(cw * 1.3))
    
    pro_scores.append(r.get("pro_score", 50))
    con_scores.append(r.get("con_score", 50))
    
    pro_evidence.append(count_evidence(p_msg))
    con_evidence.append(count_evidence(c_msg))

# Display Table
table = "| Round | Pro Words | Con Words | Pro Evidence | Con Evidence | Est. Tokens |\n"
table += "|---|---|---|---|---|---|\n"
for i in range(len(rounds)):
    toks = pro_tokens[i] + con_tokens[i]
    table += f"| {round_numbers[i]} | {pro_words[i]} | {con_words[i]} | {pro_evidence[i]} | {con_evidence[i]} | {toks} |\n"

display(Markdown(table))


In [ ]:
# Chart 1: Score Progression
plt.figure(figsize=(8, 4), dpi=100)
plt.plot(round_numbers, pro_scores, marker='o', label='PRO', color='red')
plt.plot(round_numbers, con_scores, marker='o', label='CON', color='blue')
plt.title("Persuasion Score Progression")
plt.xlabel("Round")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Chart 2: Token Usage per Round
plt.figure(figsize=(8, 4), dpi=100)
plt.bar(round_numbers, pro_tokens, label='PRO Tokens', color='red', alpha=0.7)
plt.bar(round_numbers, con_tokens, bottom=pro_tokens, label='CON Tokens', color='blue', alpha=0.7)
plt.title("Token Usage Per Round")
plt.xlabel("Round")
plt.ylabel("Tokens")
plt.legend()
plt.show()


In [ ]:
# Chart 3: Argument Word Count Comparison
import numpy as np
plt.figure(figsize=(8, 4), dpi=100)
x = np.arange(len(round_numbers))
width = 0.35
plt.bar(x - width/2, pro_words, width, label='PRO', color='red', alpha=0.7)
plt.bar(x + width/2, con_words, width, label='CON', color='blue', alpha=0.7)
plt.xticks(x, round_numbers)
plt.title("Argument Word Count Per Round")
plt.xlabel("Round")
plt.ylabel("Word Count")
plt.legend()
plt.show()


In [ ]:
# Chart 4: Evidence/Citations Count
plt.figure(figsize=(8, 4), dpi=100)
plt.plot(round_numbers, pro_evidence, marker='s', label='PRO Evidence', color='red', linestyle='--')
plt.plot(round_numbers, con_evidence, marker='s', label='CON Evidence', color='blue', linestyle='--')
plt.title("Evidence/Citations Count Per Round")
plt.xlabel("Round")
plt.ylabel("Count")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Aggregate Metrics
print("=== Aggregate Metrics ===")
print(f"Winner: {verdict.get('winner', 'Tie').upper()}")
print(f"Final PRO Score: {verdict.get('pro_score')}")
print(f"Final CON Score: {verdict.get('con_score')}")
print(f"Total Rounds: {verdict.get('round_count')}")
print(f"Total Cost: ${total_cost:.4f}")
cost_per_arg = total_cost / (verdict.get('round_count', 1) * 2)
print(f"Cost per argument: ${cost_per_arg:.4f}")


In [ ]:
# Cost Projection Table
proj = "| Debates | Estimated Cost |\n"
proj += "|---|---|\n"
for scale in [1, 10, 100, 1000]:
    proj += f"| {scale} | ${total_cost * scale:.2f} |\n"
display(Markdown(proj))


## Analysis & Interpretation
**Verdict & Why:**
The CON side ultimately won the debate, achieving a final score of 51.43 to PRO's 48.57. As the verdict highlighted, CON presented empirically verifiable claims (e.g., specific charitable giving statistics like 91% vs 66%) that survived demographic adjustments. While PRO made sophisticated methodological critiques about confounding variables, they failed to provide affirmative empirical evidence to support those counterclaims.

**Token Usage & Context Accumulation:**
Token usage scales significantly as rounds progress because the agents must consume the entire prior context (their own previous messages and their opponent's messages) to formulate coherent rebuttals. The stacked bar chart illustrates how the LLM context window fills up, which directly drives up the API cost in later rounds.

**Cost-Per-Insight Tradeoff (Haiku/Sonnet):**
This debate cost approximately $0.16. At an average cost of ~$0.027 per argument, utilizing smaller, faster models like Claude 3 Haiku for the initial rounds of a debate could drastically reduce the cost-per-insight while reserving more expensive models (like Sonnet or Opus) for the final summations. The cost projection shows that running 1,000 such debates would cost ~$166, emphasizing the need for strategic model selection in large-scale multi-agent deployments.